# Day 14 — multi-hop retrieval (sufficiency check + follow-up search)
The frozen within-paper agent (Day 7 v2) plans all its searches up front. The multi-hop agent adds a loop:

    decompose -> retrieve -> check --(enough evidence)----------------------> answer
                                \--(something missing)--> follow_up --> (check again if hops left) -> answer

* **check** (LLM): reads the question and short previews of the 7 evidence passages; says whether they are enough,
  and if not, names what is missing and writes ONE new search query (in the paper's own words).
* **follow_up**: searches the paper with that query and adds up to 3 NEW passages (E8-E10). 1 extra hop at most.
* decompose / retrieve / answer are identical to the frozen agent, so any change comes from the extra hop.

**Evaluation: DEV, the same 50 questions as Day 7-8 (seed 0)**, compared with the frozen v2 run on them.
The 100-question test result stays frozen; this is reported as a dev-set extension.
**Quota:** ~4.3k gpt-oss-120b tokens/question (~215k for 50, a bit more than one day): run with `--resume` over 2 days.

In [1]:
# WHAT: Mounts Google Drive and makes the research-engine folder the working directory.
# WHY:  All commands below run from the project root. Run this first in every new session.
from google.colab import drive
drive.mount('/content/drive')

import os, glob
PROJECT = glob.glob('/content/drive/MyDrive/research-engine')[0]
os.chdir(PROJECT)
print(os.getcwd())

Mounted at /content/drive
/content/drive/MyDrive/research-engine


In [2]:
# WHAT: Installs the project's packages (run again after every runtime restart).
# WHY:  A fresh Colab session doesn't have them.
!pip install -q -r requirements.txt pytest

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 96.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 15.1 MB/s eta 0:00:00


In [3]:
# WHAT: Copies the Groq key from Colab Secrets into an environment variable and checks for a GPU.
# WHY:  The `!python` commands can only see environment variables; the reranker needs the GPU to be fast.
from google.colab import userdata
import torch
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY").strip()
print("key loaded:", len(os.environ["GROQ_API_KEY"]) > 0, "| GPU:", torch.cuda.is_available())

key loaded: True | GPU: True


In [ ]:
# WHAT: Changes one import line in tests/test_multihop.py.
# WHY:  Colab has another package called "tests" installed, so "from tests.test_agents import ..." found that
#       package instead of ours. Importing "test_agents" directly avoids the name clash (works in CI too).
!sed -i 's/^from tests.test_agents import FakePipeline/from test_agents import FakePipeline/' tests/test_multihop.py
!grep -n "import FakePipeline" tests/test_multihop.py

5:from test_agents import FakePipeline


In [ ]:
# WHAT: Runs the whole test suite (now 52 tests, incl. the multi-hop graph with a fake LLM).
# WHY:  Confirms the new code works and nothing old broke, before spending quota. Expect "52 passed".
!python -m pytest -q

....................................................                     [100%]
52 passed in 1.20s


## Step 1 — one question, to see a hop happen

In [ ]:
# WHAT: Runs the multi-hop agent on dev question 30 of the n=50 sample ("what datasets did they use?"), a Day 8
#       retrieval miss (evidence recall 0.00), and prints the check's verdict, the follow-up query and the answer.
# WHY:  To read one real hop before running all 50. Costs one question (~4-5k tokens).
import random
from agent.multihop import MultiHopAgent
from retrieval.data import load_questions
from retrieval.pipeline import get_pipeline
pipe = get_pipeline()
sample = random.Random(0).sample(load_questions("dev", set(pipe.chunk_ids)), 50)
q = sample[29]
out = MultiHopAgent(pipeline=pipe).run(q["question"], q["paper_id"])
print("Q30:", q["question"], "| paper", q["paper_id"])
print("sufficient:", out["sufficient"], "| missing:", out["missing"], "| follow-up query:", out["follow_up_query"])
for e in out["evidence"]:
    gold = "GOLD" if e["chunk_id"] in q["gold"] else "    "
    print(f"  [{e['label']}] hop{e['hop']} {gold} ({e['section'][:25]}) {e['text'][:90]}")
print("\nAnswer:\n" + out["answer"])
print("\nreference:", q.get("reference_answers", [])[:2])
print("tokens:", out["tokens"], "|", out["secs"], "s")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Q30: what datasets did they use? | paper 1911.03059
sufficient: False | missing: dataset name | follow-up query: what dataset was used for training and validation
  [E1] hop0      (Implementation of the Sys) We use two setups (as done in [ref]) for our system. In the first setup, we eliminate the 
  [E2] hop0      (Implementation of the Sys) Table II shows the accuracy and F1 score for different classifiers with and without elimin
  [E3] hop0      (Related Works ::: Researc) Researches on question classification, question taxonomies and QA system have been underta
  [E4] hop0      (Proposed Methodology) We use different types of classifiers for QA type classification. We separate our methodol
  [E5] hop0      (Related Works ::: Researc) A Bengali QC System was built by Somnath Banerjee and Sivaji Bandyopadhyay [ref] [ref] [re
  [E6] hop0      (Implementation of the Sys) Gradient Boosting Classifier produces a prediction model consisting of weak prediction mod
  [E7] hop0      (Introduc

## Step 2 — the 50 dev questions
Saves one line per question to `results/agent_eval_dev_n50_v3hop.jsonl`. If Groq's daily limit is reached it prints
**STOPPED** and keeps everything; run the same cell again the next day (`--resume`).

In [4]:
# WHAT: Runs the multi-hop agent on the same 50 dev questions as the frozen v2 run (seed 0).
# WHY:  ev_rec = gold passages that reached the LLM; hops=1 means the check asked for a follow-up search, and
#       ev_rec_1st shows the recall before that hop. ~4.3k tokens/question.
!python -m eval.run_agent_eval --n 50 --tag v3hop --multihop --resume

modules.json: 100% 349/349 [00:00<00:00, 1.58MB/s]
config_sentence_transformers.json: 100% 124/124 [00:00<00:00, 564kB/s]
README.md: 100% 94.8k/94.8k [00:00<00:00, 76.0MB/s]
sentence_bert_config.json: 100% 52.0/52.0 [00:00<00:00, 279kB/s]
config.json: 100% 743/743 [00:00<00:00, 3.98MB/s]

model.safetensors: downloading bytes:  64% 85.7M/133M [00:00<00:00, 125MB/s, 6.08MB/s  ] 
model.safetensors: downloading bytes: 100% 86.7M/86.7M [00:01<00:00, 70.6MB/s, 8.31MB/s  ]
model.safetensors: reconstructing file: 100% 133M/133M [00:01<00:00, 109MB/s, 13.0MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 5098.17it/s]
tokenizer_config.json: 100% 366/366 [00:00<00:00, 2.28MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 28.9MB/s]
tokenizer.json: 100% 711k/711k [00:00<00:00, 66.3MB/s]
special_tokens_map.json: 100% 125/125 [00:00<00:00, 826kB/s]
config.json: 100% 190/190 [00:00<00:00, 931kB/s]
Loading cached embeddings: /content/drive/MyDrive/research-engine/cache/emb_BAAI_bge-small-en-v1.5_28541.np

In [5]:
# WHAT: Counts the questions saved so far.
# WHY:  If the output above looks frozen, check progress here before assuming the run is stuck.
!wc -l results/agent_eval_dev_n50_v3hop.jsonl

50 results/agent_eval_dev_n50_v3hop.jsonl


## Step 3 — the judge (gpt-oss-20b, its own quota)

In [6]:
# WHAT: Judges each multi-hop answer: citation support, faithfulness, correctness against the references.
# WHY:  Same judge and prompt as the v2 dev run, so the two can be compared directly.
!python -m eval.judge_agent_eval results/agent_eval_dev_n50_v3hop.jsonl --resume

50 answers in agent_eval_dev_n50_v3hop.jsonl | judge=openai/gpt-oss-20b | resuming: 41 already judged
[ 42/50] strict=1.00 judged=1.00 faithful=yes correct=yes  How they extract "structured answer-relevant relation"?
[ 43/50] strict=1.00 judged=1.00 faithful=yes correct=yes  How are the accuracy merits of the approach demonstrate
[ 44/50] strict=0.67 judged=0.50 faithful=yes correct=partly  What metrics are used for evaluation?
[ 45/50] strict=0.00 judged=1.00 faithful=yes correct=partly  What are the baselines?
[ 46/50] strict=1.00 judged=1.00 faithful=yes correct=yes  What sentiment analysis dataset is used?
[ 47/50] strict=0.00 judged=0.00 faithful=no correct=yes  did they compare with other extractive summarization me
[ 48/50] strict=1.00 judged=1.00 faithful=yes correct=yes  How do their models decide how much improtance to give 
[ 49/50] strict=0.00 judged=0.50 faithful=partly correct=no  What kind of issues (that are not on the forefront of c
[ 50/50] strict=0.00 judged=1.00 fai

## Step 4 — compare with the frozen agent (v2) on the same 50 questions (no LLM)

In [7]:
# WHAT: Puts v2 (frozen) and v3hop side by side: retrieval, citations, judge scores, cost, plus how often it hopped.
# WHY:  The numbers for the results cell, the report and the CV.
import json
def summ(f, key="summary"):
    return json.load(open(f))[key] if os.path.exists(f) else {}
pairs = [("results/agent_eval_dev_n50_v2.json", "results/agent_eval_dev_n50_v3hop.json"),
         ("results/agent_eval_dev_n50_v2_judge-gpt-oss-20b.json", "results/agent_eval_dev_n50_v3hop_judge-gpt-oss-20b.json")]
for a, b in pairs:
    A, B = summ(a), summ(b)
    print(f"\n{a.split('/')[-1]}  vs  v3hop")
    print(f"  {'metric':<44} {'v2':>8} {'v3hop':>8}")
    for k in list(A) + [k for k in B if k not in A]:
        print(f"  {k:<44} {str(A.get(k, '-')):>8} {str(B.get(k, '-')):>8}")


agent_eval_dev_n50_v2.json  vs  v3hop
  metric                                             v2    v3hop
  citation_precision                             0.6278   0.6232
  citation_hit                                      0.7      0.7
  citation_recall                                0.5467   0.5467
  evidence_recall                                0.9117    0.925
  single_query_recall                             0.905    0.905
  no_citation_rate                                  0.1     0.08
  declined_rate                                     0.1     0.08
  invalid_citation_rate                             0.0      0.0
  avg_sub_questions                                1.44     1.48
  avg_prompt_tokens                                2884     4216
  avg_completion_tokens                             186      269
  avg_secs                                        18.24    28.45
  hop_rate                                            -      0.4
  avg_evidence_chunks                              

In [8]:
# WHAT: Lists every question where the agent searched again: what was missing, the follow-up query, how many new
#       passages it added, and evidence recall before -> after the hop; then the Day 8 retrieval misses.
# WHY:  Shows whether hops find real missing evidence or are wasted calls. No LLM call.
import json
rows = [json.loads(l) for l in open("results/agent_eval_dev_n50_v3hop.jsonl")]
v2 = {r["question"]: r for r in map(json.loads, open("results/agent_eval_dev_n50_v2.jsonl"))}
for i, r in enumerate(rows, 1):
    if r.get("hops"):
        m = r["metrics"]
        print(f"Q{i:<3} ev_rec {m['evidence_recall_first_pass']:.2f} -> {m['evidence_recall']:.2f} "
              f"| +{len(r['hop_log'][0]['added_chunk_ids'])} | missing: {r['missing'][:60]} | query: {r['follow_up_query'][:60]}")
print("\nDay 8 retrieval misses (v2 ev_rec -> v3hop ev_rec):")
for i in (30, 33, 45, 47, 49):
    if i <= len(rows):
        r = rows[i - 1]
        print(f"  Q{i}: {v2[r['question']]['metrics']['evidence_recall']:.2f} -> {r['metrics']['evidence_recall']:.2f}"
              f"  hops={r.get('hops')}  {r['question'][:60]}")

Q2   ev_rec 1.00 -> 1.00 | +3 | missing: methodology for ensuring lack of bias in dialect, topic, and | query: how did the authors sample tweets to avoid bias toward speci
Q10  ev_rec 1.00 -> 1.00 | +3 | missing: distribution statistics of retweet counts and exposure times | query: retweet count distribution fake news versus non‑fake news Tw
Q13  ev_rec 1.00 -> 1.00 | +3 | missing: language of dataset | query: dataset language English
Q14  ev_rec 1.00 -> 1.00 | +3 | missing: details of how the proposed metric Sera measures content rel | query: Sera content relevance measurement method
Q15  ev_rec 1.00 -> 1.00 | +3 | missing: explanation of why back-translation improves performance | query: explanation effectiveness of back-translation
Q16  ev_rec 1.00 -> 1.00 | +3 | missing: annotation procedure details | query: annotation guidelines hierarchical depression symptoms datas
Q18  ev_rec 1.00 -> 1.00 | +3 | missing: number of participants in the human evaluation study | query: human evalua

In [9]:
# WHAT: Splits the v2 -> v3hop correctness change into questions that made a follow-up search and questions that
#       didn't, and counts how many answers actually cite a passage found by the follow-up search (E8-E10).
# WHY:  Evidence recall shows the hop never added a QASPER-gold passage, yet correctness rose 0.68 -> 0.77. If the
#       gain sits in hopped questions that cite hop passages, the hop helped (gold labels just miss it); if it is
#       spread evenly or sits in non-hopped questions, it is run-to-run noise. No LLM call.
import json
SC = {"yes": 1.0, "partly": 0.5, "no": 0.0}
def load(p): return [json.loads(l) for l in open(p)]
v3 = load("results/agent_eval_dev_n50_v3hop.jsonl")
j3 = {j["question"]: j for j in load("results/agent_eval_dev_n50_v3hop_judge-gpt-oss-20b.jsonl")}
j2 = {j["question"]: j for j in load("results/agent_eval_dev_n50_v2_judge-gpt-oss-20b.jsonl")}
groups = {"hopped": [], "not hopped": []}
for i, r in enumerate(v3, 1):
    hop_ids = {c for c, h in zip(r["evidence_chunk_ids"], r.get("evidence_hops", [])) if h}
    cites_hop = any(c in hop_ids for c in r["cited_chunk_ids"])
    c2 = SC.get(j2[r["question"]]["judge"]["correct"], 0.0)
    c3 = SC.get(j3[r["question"]]["judge"]["correct"], 0.0)
    groups["hopped" if r.get("hops") else "not hopped"].append((i, c2, c3, cites_hop))
for g, xs in groups.items():
    n = len(xs)
    print(f"{g:<11} n={n:<3} correctness v2 {sum(x[1] for x in xs)/n:.3f} -> v3hop {sum(x[2] for x in xs)/n:.3f}"
          + (f" | answers citing a hop passage: {sum(x[3] for x in xs)}" if g == "hopped" else ""))
print("\nQuestions whose correctness changed (Q: v2 -> v3hop, hopped?, cites hop passage?):")
for g, xs in groups.items():
    for i, c2, c3, ch in xs:
        if c2 != c3:
            print(f"  Q{i:<3} {c2} -> {c3}  {'hopped' if g == 'hopped' else '      '}  {'cites E8+' if ch else ''}")

hopped      n=20  correctness v2 0.550 -> v3hop 0.625 | answers citing a hop passage: 0
not hopped  n=30  correctness v2 0.767 -> v3hop 0.867

Questions whose correctness changed (Q: v2 -> v3hop, hopped?, cites hop passage?):
  Q10  0.0 -> 1.0  hopped  
  Q45  0.0 -> 0.5  hopped  
  Q5   0.5 -> 1.0          
  Q11  1.0 -> 0.5          
  Q26  0.0 -> 1.0          
  Q40  0.5 -> 1.0          
  Q41  0.0 -> 1.0          
  Q44  1.0 -> 0.5          
  Q48  0.0 -> 1.0          


## Results — multi-hop (dev, same 50 questions as v2; one run, prompts not tuned)
* Evidence recall 0.912 (v2) -> 0.925; on the 20 questions that hopped, recall before = after the hop (0.925): the follow-up search never added a QASPER-gold passage (Q45's 0.00 -> 1.00 already came from the first search in this run).
* Judge (gpt-oss-20b): citation support 0.879 -> 0.897, faithfulness 0.878 -> 0.880, correctness 0.68 -> 0.77 (answered-only 0.756 -> 0.837).
* The correctness gain is NOT from the hop: 0 of 20 hopped answers cite a passage found by the follow-up search, and correctness rose more on the 30 questions that did not hop (0.767 -> 0.867) than on the hopped ones (0.550 -> 0.625). It is run-to-run variation (answers, decomposition and judge regenerated): about ±10 points at n=50.
* Cost: hop rate 40%, 4,216 + 269 tokens/question (+46% vs 2,884 + 186), 28.5 s vs 18.2 s.
* Why it does not help: first-pass recall is already 0.91+, so the check mostly asks for details the paper does not contain (Q31 number of EEG electrodes) or that are already present; on the real vocabulary-gap miss (Q30) the follow-up query rewords the question instead of using the paper's terms.
* Decision: the single-pass agent (Day 7 v2) stays the default; multi-hop is kept as an option (app switch, `--multihop`) and reported as a negative ablation.